# Лабораторная работа 2

## Метод ветвей и границ для задачи о максимальной клике

В этой лабораторной работе нужно реализовать алгоритм для задачи о **максимальной клике**.

Граф состоит из вершин и рёбер. Клика - это такое множество вершин, где каждая вершина соединена ребром с каждой другой вершиной из этого множества.

Например, если вершины $1$, $2$, $3$ все попарно соединены, то множество $\{1, 2, 3\}$ является кликой.

Задача максимальной клики:

$$
\text{найти клику максимального размера}
$$

То есть нужно найти как можно больше вершин, которые все попарно соединены между собой.

В этой работе используется математическая модель через переменные $x_i$ и собственная реализация метода ветвей и границ. Солвер HiGHS используется только для решения ЛП-релаксаций.

# 1. Математическая постановка задачи

Для каждой вершины графа вводим бинарную переменную:

$$
x_i \in \{0, 1\}
$$

Если вершина $i$ входит в клику, то:

$$
x_i = 1
$$

Если вершина $i$ не входит в клику, то:

$$
x_i = 0
$$

Размер выбранного множества вершин равен:

$$
\sum_{i = 1}^{n} x_i
$$

Поэтому целевая функция:

$$
\max \sum_{i = 1}^{n} x_i
$$

Главное ограничение: если две вершины $i$ и $j$ не соединены ребром, то нельзя взять их обе в клику.

Для такой пары добавляется ограничение:

$$
x_i + x_j \le 1
$$

Почему это работает:

$$
\begin{aligned}
x_i = 0,\ x_j = 0 &\Rightarrow x_i + x_j = 0 \le 1 \\
x_i = 1,\ x_j = 0 &\Rightarrow x_i + x_j = 1 \le 1 \\
x_i = 0,\ x_j = 1 &\Rightarrow x_i + x_j = 1 \le 1 \\
x_i = 1,\ x_j = 1 &\Rightarrow x_i + x_j = 2 > 1
\end{aligned}
$$

То есть ограничение запрещает только один плохой случай - когда две несоединённые вершины одновременно попали в клику.

# 2. Усиление модели через независимые множества

Если есть группа вершин, между которыми нет ни одного ребра, то такая группа называется независимым множеством.

Пусть есть независимое множество:

$$
S = \{v_1, v_2, \ldots, v_k\}
$$

Так как внутри этого множества никакие две вершины не соединены, то в клику можно взять максимум одну вершину из $S$.

Поэтому вместо многих парных ограничений можно добавить одно сильное ограничение:

$$
\sum_{v_i \in S} x_{v_i} \le 1
$$

Это делает ЛП-релаксацию сильнее.

Например, если использовать только парные ограничения, дробное решение может быть таким:

$$
x_1 = 0.5,\quad x_2 = 0.5,\quad x_3 = 0.5,\quad x_4 = 0.5
$$

Парные ограничения вида $x_i + x_j \le 1$ могут выполняться, но сумма будет слишком большой:

$$
x_1 + x_2 + x_3 + x_4 = 2
$$

А ограничение по независимому множеству сразу запрещает это:

$$
x_1 + x_2 + x_3 + x_4 \le 1
$$

# 3. Что такое ЛП-релаксация

Исходная задача целочисленная:

$$
x_i \in \{0, 1\}
$$

Но в каждой вершине метода ветвей и границ мы решаем более простую задачу - ЛП-релаксацию:

$$
0 \le x_i \le 1
$$

Теперь переменные могут быть дробными:

$$
x_i = 0.3,\quad x_j = 0.7
$$

Такое решение не является настоящей кликой, но оно даёт верхнюю оценку.

Если ЛП-релаксация дала значение:

$$
27.8
$$

то настоящая целочисленная клика в этой ветви не может быть больше:

$$
\lfloor 27.8 \rfloor = 27
$$

Если у нас уже есть клика размера $30$, то такую ветвь можно отбросить.

# 4. Метод ветвей и границ

Метод ветвей и границ работает так:

1. Решаем ЛП-релаксацию.
2. Если задача несовместна, отбрасываем ветвь.
3. Если верхняя оценка не лучше текущей найденной клики, отбрасываем ветвь.
4. Если решение целое, проверяем, что это действительно клика.
5. Если решение дробное, выбираем дробную переменную $x_k$.
6. Создаём две новые ветви:

$$
x_k = 1
$$

и:

$$
x_k = 0
$$

Важно: программа должна честно проверить все ветви. Она может завершиться по таймауту, но тогда должна написать, что оптимальность не доказана.

# 5. Установка зависимостей

В ноутбуке используется пакет `highspy`.

Если он ещё не установлен, раскомментируй и запусти следующую ячейку.

In [ ]:
# %pip install highspy

# 6. Импорты и структуры данных

`Graph` хранит граф.

Поле `adj` хранит список битовых масок. Для каждой вершины $u$ в `adj[u]` записано, с какими вершинами она соединена.

Например, если в бите $v$ стоит $1$, значит есть ребро:

$$
(u, v)
$$

`Node` хранит одну вершину дерева Branch and Bound.

В ней есть нижние и верхние границы для переменных:

$$
l_i \le x_i \le u_i
$$

Если мы хотим зафиксировать переменную $x_k = 1$, то ставим:

$$
l_k = 1,\quad u_k = 1
$$

Если хотим зафиксировать $x_k = 0$, то ставим:

$$
l_k = 0,\quad u_k = 0
$$

In [ ]:
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import highspy


EPS = 1e-7


@dataclass
class Graph:
  n: int
  adj: List[int]
  edges_count: int


@dataclass
class Node:
  lower: List[float]
  upper: List[float]
  depth: int


@dataclass
class LpResult:
  status: str
  objective: float
  values: List[float]


@dataclass
class SolverStats:
  nodes: int = 0
  pruned_by_bound: int = 0
  pruned_by_infeasible: int = 0
  integer_solutions: int = 0
  max_depth: int = 0

# 7. Чтение графа из DIMACS `.clq`

Строка `p edge 125 6963` означает:

$$
n = 125
$$

$$
m = 6963
$$

Строка `e 1 2` означает, что есть ребро между вершинами $1$ и $2$.

В файле вершины нумеруются с $1$, а в Python удобнее хранить с $0$, поэтому при чтении делаем:

$$
u = u_{\text{file}} - 1
$$

$$
v = v_{\text{file}} - 1
$$

In [ ]:
def read_dimacs_clq(path: str) -> Graph:
  n = 0
  edges_count = 0
  raw_edges: List[Tuple[int, int]] = []

  with open(path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
      line = line.strip()

      if not line or line.startswith("c"):
        continue

      parts = line.split()

      if parts[0] == "p":
        n = int(parts[2])
        edges_count = int(parts[3])
        continue

      if parts[0] == "e":
        u = int(parts[1]) - 1
        v = int(parts[2]) - 1

        if u == v:
          continue

        raw_edges.append((u, v))

  if n <= 0:
    raise ValueError("DIMACS file does not contain a valid 'p edge n m' line")

  adj = [0 for _ in range(n)]

  for u, v in raw_edges:
    if not (0 <= u < n and 0 <= v < n):
      raise ValueError(f"edge contains invalid vertex index: {u + 1}, {v + 1}")

    adj[u] |= 1 << v
    adj[v] |= 1 << u

  return Graph(n=n, adj=adj, edges_count=len(raw_edges) if edges_count == 0 else edges_count)

# 8. Вспомогательные функции для графа

`has_edge` проверяет, есть ли ребро между вершинами $u$ и $v$.

Так как граф хранится через битовые маски, проверка работает быстро:

$$
\text{adj}[u]\ \&\ (1 \ll v)
$$

Если нужный бит равен $1$, значит ребро есть.

`bit_count` считает количество единичных битов. В нашем случае это степень вершины, то есть количество соседей.

In [ ]:
def has_edge(adj: List[int], u: int, v: int) -> bool:
  return ((adj[u] >> v) & 1) == 1


def bit_count(x: int) -> int:
  return x.bit_count()

# 9. Построение независимых множеств

Здесь мы строим усиленные ограничения.

Идея:

1. Перебираем пару вершин $u$ и $v$.
2. Если между ними есть ребро, то это не конфликтующая пара, пропускаем.
3. Если между ними нет ребра, начинаем строить независимое множество.
4. Добавляем к нему другие вершины $w$, которые не имеют рёбер ни с одной уже добавленной вершиной.
5. Для полученной группы добавляем ограничение:

$$
\sum_{i \in S} x_i \le 1
$$

Массив `covered` нужен, чтобы не строить одинаковые ограничения слишком много раз.

In [ ]:
def build_independent_set_constraints(graph: Graph) -> List[List[int]]:
  n = graph.n
  adj = graph.adj

  covered = [0 for _ in range(n)]
  constraints: List[List[int]] = []

  for u in range(n):
    for v in range(u + 1, n):
      if has_edge(adj, u, v):
        continue

      if (covered[u] >> v) & 1:
        continue

      mask = (1 << u) | (1 << v)
      group = [u, v]

      for w in range(n):
        if w == u or w == v:
          continue

        if adj[w] & mask:
          continue

        group.append(w)
        mask |= 1 << w

      for i in range(len(group)):
        a = group[i]

        for j in range(i + 1, len(group)):
          b = group[j]
          covered[a] |= 1 << b
          covered[b] |= 1 << a

      constraints.append(group)

  return constraints

# 10. Начальная клика жадным алгоритмом

Перед Branch and Bound полезно быстро найти хотя бы какую-то клику.

Это не обязательно оптимальная клика, но она даёт начальную нижнюю границу.

Если жадный алгоритм нашёл клику размера $20$, то все ветви, где верхняя оценка не больше $20$, можно сразу отбрасывать.

Алгоритм:

1. Сортируем вершины по степени.
2. Берём стартовую вершину.
3. Пытаемся добавлять только те вершины, которые соединены со всеми уже выбранными.
4. Запоминаем лучшую найденную клику.

In [ ]:
def greedy_initial_clique(graph: Graph) -> List[int]:
  n = graph.n
  order = sorted(range(n), key=lambda v: bit_count(graph.adj[v]), reverse=True)

  best: List[int] = []

  for start in order:
    clique = [start]
    candidates = graph.adj[start]

    for v in order:
      if v == start:
        continue

      if ((candidates >> v) & 1) == 0:
        continue

      clique.append(v)
      candidates &= graph.adj[v]

    if len(clique) > len(best):
      best = clique

  return sorted(best)

# 11. Решение ЛП-релаксации через HiGHS

В этой функции создаётся задача:

$$
\max \sum_{i = 1}^{n} x_i
$$

при ограничениях:

$$
\sum_{i \in S} x_i \le 1
$$

и границах:

$$
l_i \le x_i \le u_i
$$

В корневой вершине:

$$
0 \le x_i \le 1
$$

В дочерних вершинах некоторые переменные фиксируются:

$$
x_k = 0
$$

или:

$$
x_k = 1
$$

HiGHS возвращает дробное решение и значение целевой функции. Это значение используется как верхняя граница для ветви.

In [ ]:
def solve_lp_relaxation(
  graph: Graph,
  constraints: List[List[int]],
  lower: List[float],
  upper: List[float],
  simplex: bool,
) -> LpResult:
  h = highspy.Highs()
  h.setOptionValue("output_flag", False)

  if simplex:
    h.setOptionValue("solver", "simplex")

  h.addVars(graph.n, lower, upper)
  h.changeObjectiveSense(highspy.ObjSense.kMaximize)
  h.changeColsCost(graph.n, list(range(graph.n)), [1.0] * graph.n)

  inf = highspy.kHighsInf

  for group in constraints:
    h.addRow(
      -inf,
      1.0,
      len(group),
      group,
      [1.0] * len(group),
    )

  h.run()

  status = h.modelStatusToString(h.getModelStatus())

  if "Infeasible" in status:
    return LpResult(status=status, objective=-math.inf, values=[])

  if "Optimal" not in status:
    return LpResult(status=status, objective=-math.inf, values=[])

  solution = h.getSolution()
  values = list(solution.col_value)
  objective = float(h.getObjectiveValue())

  return LpResult(status=status, objective=objective, values=values)

# 12. Проверка целочисленности решения

ЛП-релаксация может вернуть дробные значения:

$$
x_i = 0.5
$$

$$
x_j = 0.8
$$

Такое решение не является настоящей кликой.

Если все переменные близки к $0$ или $1$, то решение можно считать целым.

Из-за численных погрешностей мы сравниваем не строго с $0$ и $1$, а с точностью `EPS`.

In [ ]:
def get_integral_solution(values: List[float]) -> Optional[List[int]]:
  result: List[int] = []

  for value in values:
    if abs(value) <= EPS:
      result.append(0)
      continue

    if abs(value - 1.0) <= EPS:
      result.append(1)
      continue

    return None

  return result

# 13. Выбор переменной для ветвления

Если решение дробное, нужно выбрать переменную $x_k$ и создать две ветви:

$$
x_k = 1
$$

и:

$$
x_k = 0
$$

Здесь выбирается переменная, которая ближе всего к $0.5$.

Почему это разумно: такая переменная наиболее неопределённая. Если $x_i = 0.99$, то она почти уже равна $1$. Если $x_i = 0.5$, то солвер сам не может понять, брать её или не брать.

In [ ]:
def choose_branch_variable(values: List[float]) -> Optional[int]:
  best_index = None
  best_score = math.inf

  for i, value in enumerate(values):
    if EPS < value < 1.0 - EPS:
      score = abs(value - 0.5)

      if score < best_score:
        best_score = score
        best_index = i

  return best_index

# 14. Проверка найденной клики

Это обязательный чекер.

Программа не должна просто доверять себе. Если она говорит, что нашла клику, нужно проверить все пары вершин внутри найденного множества.

Для любой пары $u$ и $v$ внутри клики должно быть ребро:

$$
(u, v) \in E
$$

Если хотя бы одной пары нет, ответ некорректный.

In [ ]:
def is_clique(graph: Graph, vertices: List[int]) -> bool:
  for i in range(len(vertices)):
    u = vertices[i]

    for j in range(i + 1, len(vertices)):
      v = vertices[j]

      if not has_edge(graph.adj, u, v):
        return False

  return True

# 15. Основной алгоритм Branch and Bound

Это главная часть программы.

Переменная `best_clique` хранит лучшую найденную клику.

Стек `stack` хранит ещё не проверенные ветви.

Каждая ветвь должна быть обработана одним из способов:

1. Отброшена как несовместная.
2. Отброшена по верхней границе.
3. Дала целое решение.
4. Разбита на две новые ветви.

Если стек опустел, значит все ветви проверены и оптимальность доказана.

Если вышел таймаут, программа завершится с:

```text
proven_optimal: False
```

Это означает: найденная клика корректная, но оптимальность не доказана.

In [ ]:
def branch_and_bound(
  graph: Graph,
  constraints: List[List[int]],
  timeout_sec: float,
  simplex: bool,
) -> Tuple[List[int], bool, SolverStats]:
  start_time = time.perf_counter()
  stats = SolverStats()

  best_clique = greedy_initial_clique(graph)
  best_size = len(best_clique)

  root = Node(
    lower=[0.0] * graph.n,
    upper=[1.0] * graph.n,
    depth=0,
  )

  stack = [root]
  proven_optimal = True

  while stack:
    if time.perf_counter() - start_time >= timeout_sec:
      proven_optimal = False
      break

    node = stack.pop()
    stats.nodes += 1
    stats.max_depth = max(stats.max_depth, node.depth)

    lp = solve_lp_relaxation(
      graph=graph,
      constraints=constraints,
      lower=node.lower,
      upper=node.upper,
      simplex=simplex,
    )

    if not lp.values:
      stats.pruned_by_infeasible += 1
      continue

    upper_bound = math.floor(lp.objective + EPS)

    if upper_bound <= best_size:
      stats.pruned_by_bound += 1
      continue

    rounded = get_integral_solution(lp.values)

    if rounded is not None:
      clique = [i for i, value in enumerate(rounded) if value == 1]

      if is_clique(graph, clique):
        stats.integer_solutions += 1

        if len(clique) > best_size:
          best_clique = clique
          best_size = len(clique)

      continue

    branch_var = choose_branch_variable(lp.values)

    if branch_var is None:
      continue

    left_lower = node.lower.copy()
    left_upper = node.upper.copy()
    left_lower[branch_var] = 1.0
    left_upper[branch_var] = 1.0

    right_lower = node.lower.copy()
    right_upper = node.upper.copy()
    right_lower[branch_var] = 0.0
    right_upper[branch_var] = 0.0

    stack.append(Node(lower=right_lower, upper=right_upper, depth=node.depth + 1))
    stack.append(Node(lower=left_lower, upper=left_upper, depth=node.depth + 1))

  return sorted(best_clique), proven_optimal, stats

# 16. Вывод результата

В результате полезно вывести:

- имя файла;
- количество вершин;
- количество рёбер;
- размер найденной клики;
- доказана ли оптимальность;
- прошла ли проверка клики;
- сколько вершин дерева обработано;
- сколько ветвей отброшено по границе;
- сколько ветвей оказалось несовместными;
- максимальную глубину дерева;
- сами вершины клики.

Вершины выводятся в нумерации с $1$, потому что в DIMACS-файлах вершины тоже нумеруются с $1$.

In [ ]:
def print_result(
  path: str,
  graph: Graph,
  clique: List[int],
  proven_optimal: bool,
  stats: SolverStats,
  elapsed: float,
) -> None:
  print(f"file: {path}")
  print(f"vertices: {graph.n}")
  print(f"edges: {graph.edges_count}")
  print(f"clique_size: {len(clique)}")
  print(f"proven_optimal: {proven_optimal}")
  print(f"checker_is_clique: {is_clique(graph, clique)}")
  print(f"nodes: {stats.nodes}")
  print(f"pruned_by_bound: {stats.pruned_by_bound}")
  print(f"pruned_by_infeasible: {stats.pruned_by_infeasible}")
  print(f"integer_solutions: {stats.integer_solutions}")
  print(f"max_depth: {stats.max_depth}")
  print(f"elapsed_sec: {elapsed:.3f}")
  print("clique_vertices_1_based:")
  print(" ".join(str(v + 1) for v in clique))

# 17. Удобная функция запуска из ноутбука

В `.py` файле удобно использовать `argparse`.

В ноутбуке удобнее сделать функцию `run_lab`, куда можно передать путь к графу.

Пример:

```python
run_lab("C125.9.clq", timeout_sec=300)
```

In [ ]:
def run_lab(path: str, timeout_sec: float = 300.0, simplex: bool = True) -> Tuple[List[int], bool, SolverStats]:
  start = time.perf_counter()

  graph = read_dimacs_clq(path)
  constraints = build_independent_set_constraints(graph)

  print(f"loaded graph: n={graph.n}, edges={graph.edges_count}")
  print(f"independent-set constraints: {len(constraints)}")

  clique, proven_optimal, stats = branch_and_bound(
    graph=graph,
    constraints=constraints,
    timeout_sec=timeout_sec,
    simplex=simplex,
  )

  elapsed = time.perf_counter() - start

  print_result(
    path=path,
    graph=graph,
    clique=clique,
    proven_optimal=proven_optimal,
    stats=stats,
    elapsed=elapsed,
  )

  if not is_clique(graph, clique):
    raise RuntimeError("checker failed: returned set is not a clique")

  return clique, proven_optimal, stats

# 18. Тест на маленьком графе прямо внутри ноутбука

Чтобы проверить, что код работает без внешних файлов, создадим маленький DIMACS-граф.

Пусть есть вершины:

$$
1, 2, 3, 4
$$

Рёбра:

$$
(1, 2),\ (1, 3),\ (2, 3),\ (3, 4)
$$

Тогда максимальная клика:

$$
\{1, 2, 3\}
$$

Её размер:

$$
3
$$

In [ ]:
small_graph_path = Path("small_test.clq")

small_graph_path.write_text(
  "\n".join([
    "c small test graph",
    "p edge 4 4",
    "e 1 2",
    "e 1 3",
    "e 2 3",
    "e 3 4",
    "",
  ]),
  encoding="utf-8",
)

clique, proven_optimal, stats = run_lab(str(small_graph_path), timeout_sec=30)

# 19. Запуск на настоящих графах

Графы уже лежат в папке `max_clique_txt`, поэтому можно указывать либо полный путь, либо короткое имя файла.

Потом укажи путь в переменной `GRAPH_PATH`.

Например:

```python
GRAPH_PATH = "max_clique_txt/DIMACS_all_ascii/C125.9.clq"
```

Ожидаемые размеры для Easy-графов:

```text
C125.9.clq        34
johnson8-2-4       4
johnson16-2-4      8
MANN_a9           16
keller4           11
hamming8-4        16
```

Если `checker_is_clique: True`, значит найденный набор вершин действительно является кликой.

Если `proven_optimal: True`, значит алгоритм успел проверить все ветви и доказал оптимальность.

In [ ]:
# GRAPH_PATH = "max_clique_txt/DIMACS_all_ascii/C125.9.clq"
# clique, proven_optimal, stats = run_lab(GRAPH_PATH, timeout_sec=300)

# 20. Что говорить на защите

Краткое объяснение:

Я решаю задачу максимальной клики через бинарную модель. Для каждой вершины есть переменная $x_i$, которая равна $1$, если вершина входит в клику, и $0$ иначе. Цель - максимизировать сумму выбранных вершин:

$$
\max \sum_i x_i
$$

Если две вершины не соединены ребром, то они не могут одновременно попасть в клику, поэтому добавляется ограничение:

$$
x_i + x_j \le 1
$$

Для усиления модели я использую независимые множества. Если внутри множества $S$ нет рёбер, то из него можно взять максимум одну вершину:

$$
\sum_{i \in S} x_i \le 1
$$

Метод ветвей и границ реализован вручную. В каждой вершине дерева я решаю ЛП-релаксацию через HiGHS. Если верхняя оценка хуже текущей найденной клики, ветвь отбрасывается. Если решение целое, проверяется, что найденный набор действительно является кликой. Если решение дробное, выбирается дробная переменная, и создаются две ветви:

$$
x_k = 0
$$

и:

$$
x_k = 1
$$

Программа завершается с доказанной оптимальностью только тогда, когда все ветви дерева обработаны или отброшены.